In [94]:
import re
import string
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from babel.numbers import number_re
from pandas.api.types import CategoricalDtype
from IPython.display import display
from datetime import datetime
from spellchecker import SpellChecker
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import pickle
from sklearn.preprocessing import MultiLabelBinarizer
import requests
import sys
from urllib.parse import urlparse

dataset_path = 'MetObjects.txt'
rd_seed = 42

In [95]:
# read both datasets if needed
df_artists = pd.read_csv('artists.csv', low_memory=False)
df_objects = pd.read_csv('objects.csv', low_memory=False)

In [96]:
def unique_difference(transformation, df_orig):
    """
    Function to find all unique domain values of transformation function.

    It is used to find all the original values, that has same representation after applying transformation.
    For example, if transformation is lowercase, then 'Coin' and 'coin' will transform to 'coin'.
    This function helps to visualize such occurrences, so we can check consistency, i.e. if all strigns are lowercased.
    :param transformation: function which transform values.
    :param df_orig: dataframe to examine.
    :return: key-value pairs, where key is the range, and value is domain of transformation function.
    """
    before_nuniques = df_orig.nunique()
    df_transformed = df_orig.apply(func=transformation)
    after_nuniques = df_transformed.nunique()
    diff = before_nuniques - after_nuniques

    # print some statistics
    print(f'Number of unique values before {transformation.__name__}: {before_nuniques}')
    print(f'Number of unique values after {transformation.__name__}: {after_nuniques}')
    print(f'Difference: {diff}')
    print(
        f'Ratio of values with different cases to all uniques before {transformation.__name__}: {diff / before_nuniques:.2f}')

    orig_uniques = df_orig.unique()

    transformation_mappings = dict()
    for original_value in orig_uniques:
        transformed = transformation(original_value)
        if transformed not in transformation_mappings:
            transformation_mappings[transformed] = set()
        transformation_mappings[transformed].add(original_value)

    # filter from key-value pairs, where only one unique value from domain is mapped to one range value.
    filtered_from_singletons = {transformed: originals for transformed, originals in transformation_mappings.items() if
                                len(originals) > 1}
    return filtered_from_singletons


def display_top_n_dict(dictionary, n: int = 5):
    """Helper function to display unique differences."""
    top_n_examples_list = [item for idx, item in enumerate(dictionary.items()) if idx < n]
    print(f'Top {n} examples:')
    display(top_n_examples_list)

In [97]:
def lowercase(value: str | None) -> str | None:
    if isinstance(value, str):
        return value.lower()
    return value


def whitespaces_reduction(value: str | None) -> str | None:
    if isinstance(value, str):
        return re.sub(r'\s+', ' ', value.strip())
    return value


def special_characters_reduction(value: str, pattern=r'[^\w\s]') -> str:
    if isinstance(value, str):
        return re.sub(pattern, '', value)
    return value

In [98]:
def introduce_data(data: pd.DataFrame | pd.Series, column=None):
    if column:
        selected_data = data[column]
    else:
        selected_data = data
    display(selected_data.describe())
    display(selected_data.info())
    if isinstance(selected_data, pd.Series):
        nans_number = selected_data.isna().sum()
        print(f'Number of NaN values: {nans_number} ({100 * nans_number / len(selected_data):.2f}%)')

# 7. Dimensions

In [210]:
df_dimensions = df_objects[['Object ID', 'Dimensions']].copy()
introduce_data(df_dimensions, 'Dimensions')
df_dimensions.head()

count                                        409898
unique                                       264986
top       sheet: 2 11/16 x 1 3/8 in. (6.9 x 3.5 cm)
freq                                           2297
Name: Dimensions, dtype: object

<class 'pandas.core.series.Series'>
RangeIndex: 484956 entries, 0 to 484955
Series name: Dimensions
Non-Null Count   Dtype 
--------------   ----- 
409898 non-null  object
dtypes: object(1)
memory usage: 3.7+ MB


None

Number of NaN values: 75058 (15.48%)


,Object ID,Dimensions
0,1,Dimensions unavailable
1,2,Dimensions unavailable
2,3,Diam. 11/16 in. (1.7 cm)
3,4,Diam. 11/16 in. (1.7 cm)
4,5,Diam. 11/16 in. (1.7 cm)


In [211]:
df_dimensions['Dimensions'].unique()

array(['Dimensions unavailable', 'Diam. 11/16 in. (1.7 cm)',
       'Diam. 1/2 in. (1.3 cm)', ...,
       'Plate: 8 3/4 × 5 7/8 in. (22.2 × 15 cm)\r\nSheet: 12 3/16 × 8 3/8 in. (31 × 21.2 cm)',
       '2 pages, 30 unnumbered leaves of plates (some color and folded) : illustrations (some color) ; Height: 11 7/16 in. (29 cm)',
       'Image: 6 5/8 × 11 3/4 in. (16.9 × 29.9 cm)\r\nPlate: 8 15/16 × 13 5/8 in. (22.7 × 34.6 cm)\r\nSheet: 10 13/16 × 15 9/16 in. (27.4 × 39.5 cm)'],
      dtype=object)

## Splitting dimensions

In [212]:
def split_dimensions(value):
    if not isinstance(value, str):
        return value
    return re.split(r'\r\n|;', value)

In [213]:
df_dimensions['Dimensions List'] = df_dimensions['Dimensions'].apply(split_dimensions)
df_dimensions_long = df_dimensions[['Object ID', 'Dimensions List']].explode(column='Dimensions List').rename(
    {'Dimensions List': 'Dimensions'}, axis='columns').reset_index(drop=True)
df_dimensions_long.head()

,Object ID,Dimensions
0,1,Dimensions unavailable
1,2,Dimensions unavailable
2,3,Diam. 11/16 in. (1.7 cm)
3,4,Diam. 11/16 in. (1.7 cm)
4,5,Diam. 11/16 in. (1.7 cm)


In [214]:
def chain_transformation_function(*transformations):
    def chain_transformation(value: str):
        if not isinstance(value, str):
            return value
        result = value
        for func in transformations:
            result = func(result)
        return result

    return chain_transformation

In [215]:
df_dimensions_long['Dimensions'] = df_dimensions_long['Dimensions'].apply(
    chain_transformation_function(lowercase, whitespaces_reduction))

## missing Values

In [216]:
df_dimensions_long.replace(['Dimensions unavailable', ''], np.NaN, inplace=True)

In [217]:
len(df_dimensions_long)

677700

## Extracting dimensions

In [218]:
df_dimensions_long[['Height', 'Width', 'Depth']] = np.NaN
df_dimensions_long['Filtered'] = False

In [219]:
df_dimensions_long.drop_duplicates(subset='Dimensions')

,Object ID,Dimensions,Height,Width,Depth,Filtered
0,1,dimensions unavailable,NaN,NaN,NaN,False
2,3,diam. 11/16 in. (1.7 cm),NaN,NaN,NaN,False
14,15,diam. 1/2 in. (1.3 cm),NaN,NaN,NaN,False
15,16,diam. 1 1/8 in. (2.9 cm),NaN,NaN,NaN,False
23,24,diam. 3/4 in. (1.9 cm),NaN,NaN,NaN,False
...,...,...,...,...,...,...
677691,900606,plate: 8 1/4 × 6 7/8 in. (20.9 × 17.5 cm),NaN,NaN,NaN,False
677695,900717,"2 pages, 30 unnumbered leaves of plates (some ...",NaN,NaN,NaN,False
677697,900748,image: 6 5/8 × 11 3/4 in. (16.9 × 29.9 cm),NaN,NaN,NaN,False
677698,900748,plate: 8 15/16 × 13 5/8 in. (22.7 × 34.6 cm),NaN,NaN,NaN,False


In [109]:
def get_non_extracted(dataframe: pd.DataFrame) -> pd.DataFrame:
    return dataframe[~dataframe['Filtered']].dropna(subset='Dimensions')

In [221]:

def pipeline_extraction_function(*extractions):
    def pipeline_extraction(row: pd.Series):
        dimensions = row['Dimensions']
        if not isinstance(dimensions, str):
            return row
        result = dimensions
        for func in extractions:
            result = func(result)
            if result is None:
                return row
        for extraction, attribute in zip(result, ['Height', 'Width', 'Depth']):
            row[attribute] = extraction
        row['Filtered'] = True
        return row

    return pipeline_extraction


def filter_fn(pattern: str, invert=False):
    pattern = re.compile(pattern)

    def filter(dimensions):
        match = pattern.match(dimensions)
        if match:
            return None if invert else dimensions
        else:
            return dimensions if invert else None

    return filter


def capture_fn(pattern: str):
    pattern = re.compile(pattern)

    def capture(dimensions):
        match = pattern.match(dimensions)
        if not match:
            return None
        return match.group(1)

    return capture


def convert_number(number_str: str, unit: str):
    try:
        number = float(number_str)
    except ValueError:
        return None
    if unit == 'm':
        return number * 100
    elif unit == 'mm':
        return number / 10
    else:
        return number


def extract_numbers_fn(pattern, n):
    pattern = re.compile(pattern)

    def extract_numbers(dimensions):
        match = pattern.match(dimensions)
        if not match:
            return None
        number = convert_number(match.group(1), match.group(2))
        if not number:
            return None
        return [number]

    return extract_numbers


def extract_diameter(numbers: list):
    return numbers * 2


def empty_pipeline_end(_):
    return []




In [206]:
diameter_filter = r'.*diam..*\(.*\).*'
units_group = r'(mm|cm|m)'
number_capture = r'(\d*\.?\d*)'
parentheses_capture = fr'.*\((.*{units_group}.*)\).*'
ending = r'([^\w]+.*\)|\))'
dimensions_capture = fr'.*\((.*{units_group}){ending}.*'
delimiter = fr'\s*[x×]\s*'
one_number_capture = fr'^\s*{number_capture}\s*{units_group}\s*$'
two_numbers_capture =
three_numbers_capture =
no_units_values = r'.*[^a-zA-Z](mm|cm|m)[^a-zA-Z].*'

In [ ]:
filter_no_metric_units_values = pipeline_extraction_function(filter_fn(no_units_values, invert=True),
                                                             empty_pipeline_end)
df_dimensions_long.update(get_non_extracted(df_dimensions_long).apply(filter_no_metric_units_values, axis='columns'))

In [227]:
parse_diameters = pipeline_extraction_function(filter_fn(diameter_filter),
                                               capture_fn(dimensions_capture),
                                               extract_numbers_fn(one_number_capture, 1),
                                               extract_diameter
                                               )
df_dimensions_long.update(get_non_extracted(df_dimensions_long).apply(parse_diameters, axis='columns'))

In [228]:
parse_single_number = pipeline_extraction_function(capture_fn(dimensions_capture),
                                                   extract_numbers_fn(one_number_capture, 1))
df_dimensions_long.update(get_non_extracted(df_dimensions_long).apply(parse_single_number, axis='columns'))

In [ ]:
parse_two_numbers = pipeline_extraction_function(capture_fn(dimensions_capture),
                                                   extract_numbers_fn(one_number_capture, 1))
df_dimensions_long.update(get_non_extracted(df_dimensions_long).apply(parse_single_number, axis='columns'))

In [ ]:
parse_three_numbers = pipeline_extraction_function(capture_fn(dimensions_capture),
                                                   extract_numbers_fn(one_number_capture, 1))
df_dimensions_long.update(get_non_extracted(df_dimensions_long).apply(parse_single_number, axis='columns'))

In [229]:
get_non_extracted(df_dimensions_long).drop_duplicates(subset='Dimensions')

,Object ID,Dimensions,Height,Width,Depth,Filtered
32,33,2 3/4 x 3 1/2 x 2 3/4 in. (7 x 8.9 x 7 cm),NaN,NaN,NaN,False
33,34,24 3/8 x 14 5/8 x 5 1/8 in. (61.9 x 37.1 x 13 cm),NaN,NaN,NaN,False
34,35,overall: 19 7/16 x 13 x 9 1/4 in. (49.4 x 33 x...,NaN,NaN,NaN,False
37,35,cover: 4 1/4 x 4 13/16 in. (10.8 x 12.2 cm),NaN,NaN,NaN,False
39,36,40 x 25 1/2 x 34 1/2 in. (101.6 x 64.8 x 87.6 cm),NaN,NaN,NaN,False
...,...,...,...,...,...,...
677690,900605,sheet: 4 7/16 × 6 1/16 in. (11.3 × 15.4 cm),NaN,NaN,NaN,False
677691,900606,plate: 8 1/4 × 6 7/8 in. (20.9 × 17.5 cm),NaN,NaN,NaN,False
677697,900748,image: 6 5/8 × 11 3/4 in. (16.9 × 29.9 cm),NaN,NaN,NaN,False
677698,900748,plate: 8 15/16 × 13 5/8 in. (22.7 × 34.6 cm),NaN,NaN,NaN,False


In [225]:
df_dimensions_long[df_dimensions_long['Filtered']].drop_duplicates(subset='Dimensions')

,Object ID,Dimensions,Height,Width,Depth,Filtered
0,1,dimensions unavailable,NaN,NaN,NaN,True
35,35,352 oz. 18 dwt. (10977 g),NaN,NaN,NaN,True
38,35,19 oz. 6 dwt. (600.1 g),NaN,NaN,NaN,True
280,304,13 ft. 9 in. x 24 ft. 2 1/2 in. x 50 ft. 9 1/2...,NaN,NaN,NaN,True
308,333,h. 4 1/4 in. (10.8),NaN,NaN,NaN,True
...,...,...,...,...,...,...
676545,897078,"262 pages : illustrations, facsimiles, portraits",NaN,NaN,NaN,True
676565,897090,"[42] pages, 127 leaves of plates : color illus...",NaN,NaN,NaN,True
676665,897777,w. 1 1/4 in. (3.2),NaN,NaN,NaN,True
677181,899200,please see attached list,NaN,NaN,NaN,True
